In [2]:
import rasterio
import numpy as np
import matplotlib.pyplot as plt
import os
from scipy.ndimage import sobel, generic_filter
%matplotlib inline

def readFile(file_path):
    try:
        with rasterio.open(file_path) as src:
            data = src.read()
            profile = src.profile
        return data, profile
    except Exception as e:
        print(f"Error reading the GeoTIFF file: {file_path} — {e}")
        return None, None

def calculate_ndvi(naip_array):
    nir = naip_array[3].astype(np.float32)
    red = naip_array[0].astype(np.float32)

    bottom = nir + red
    bottom[bottom == 0] = 1e-5

    ndvi = (nir - red) / bottom
    ndvi_scaled = ((ndvi + 1) / 2 * 255).clip(0, 255).astype(np.uint8)
    return ndvi_scaled[np.newaxis, :, :]  # Shape: (1, H, W)

def calculate_ndwi(naip_array):
    green = naip_array[1].astype(np.float32)
    nir = naip_array[3].astype(np.float32)

    bottom = green + nir
    bottom[bottom == 0] = 1e-5

    with np.errstate(divide='ignore', invalid='ignore'):
        ndwi = (green - nir) / bottom
        ndwi = np.nan_to_num(ndwi, nan=0.0, posinf=0.0, neginf=0.0)
    ndwi_scaled = ((ndwi + 1) / 2 * 255).clip(0, 255).astype(np.uint8)
    return ndwi_scaled[np.newaxis, :, :]

def calculate_roughness(dem_array, window_size=3):
    return generic_filter(dem_array, np.std, size=window_size, mode='nearest')

def calculate_slope(dem_array, cell_size=1):
    dx = sobel(dem_array, axis=1, mode='nearest') / (8.0 * cell_size)
    dy = sobel(dem_array, axis=0, mode='nearest') / (8.0 * cell_size)
    slope_rad = np.arctan(np.sqrt(dx**2 + dy**2))
    slope_deg = np.degrees(slope_rad)
    slope_scaled = ((slope_deg / 90) * 255).clip(0, 255).astype(np.uint8)
    return slope_scaled

def stackFiles(file_list):
    arrays = []
    base_profile = None

    for i, file_path in enumerate(file_list):
        data, profile = readFile(file_path)
        if data is not None:
            if i == 0:
                # NAIP: Add NDVI & NDWI
                ndvi_band = calculate_ndvi(data)
                ndwi_band = calculate_ndwi(data)
                data = np.concatenate([data, ndvi_band, ndwi_band], axis=0)

            elif i == 1:
                # DEM: Add roughness and slope
                # raw_dem = data[0].astype(np.float32)
                # roughness_band = calculate_roughness(raw_dem).clip(0, 255).astype(np.uint8)[np.newaxis, :, :]
                # slope_band = calculate_slope(raw_dem)[np.newaxis, :, :]
                data = np.concatenate([data], axis=0)

            elif i == 2:
                # Hillshade: use as-is
                data = data  # just for clarity


            arrays.append(data)

            if base_profile is None:
                base_profile = profile
        else:
            print(f"Error reading file: {file_path}")

    if arrays:
        stacked_data = np.concatenate(arrays, axis=0).astype(np.uint8)
        base_profile.update(count=stacked_data.shape[0], dtype='uint8')

        if "nodata" in base_profile and (
            base_profile["nodata"] is None or
            base_profile["nodata"] > 255 or
            base_profile["nodata"] < 0
        ):
            base_profile["nodata"] = 0

        return stacked_data, base_profile
    else:
        return None, None

# === MAIN ===

def process_raster_patches(
    namingOfNAIP,
    namingOfDEM,
    namingOfHillshade,
    namingOfSmashedFile,
    num_cols,
    num_rows
):
    for col in range(num_cols):
        for row in range(num_rows):
            naipFile = namingOfNAIP.replace("*", str(col), 1).replace("*", str(row), 1)
            DEMFile = namingOfDEM.replace("*", str(col), 1).replace("*", str(row), 1)
            HillshadeFile = namingOfHillshade.replace("*", str(col), 1).replace("*", str(row), 1)
            SmashedFile = namingOfSmashedFile.replace("*", str(col), 1).replace("*", str(row), 1)

            if not (os.path.exists(naipFile) and os.path.exists(DEMFile) and os.path.exists(HillshadeFile)):
                print(f"Skipping patch {col}-{row}: Missing one or more files.")
                continue

            print(f"Smashing patch {col}-{row} → {SmashedFile}")
            fileNames = [naipFile, DEMFile, HillshadeFile]

            stacked_data, profile = stackFiles(fileNames)

            if stacked_data is not None:
                os.makedirs(os.path.dirname(SmashedFile), exist_ok=True)
                with rasterio.open(SmashedFile, 'w', **profile) as dst:
                    dst.write(stacked_data)
                print(f"Smash successful for patch {col}-{row}")
            else:
                print(f"Failed to smash patch {col}-{row}")

if __name__ == "__main__":
    process_raster_patches(
        Validation Dataset/Upper_Willow_Creek_2023_NAIP_1.tif
    namingOfNAIP="/home/ec2-user/SageMaker/data/test/images/SCC_NAIP_1m_patch*-*.tif",
    # namingOfDEM="/home/ec2-user/SageMaker/data/test/dem/SCC_dem_patch*-*.tif",
    # namingOfHillshade="/home/ec2-user/SageMaker/data/test/hillshade/SCC_hillshade_patch*-*.tif",
    namingOfSmashedFile="/home/ec2-user/SageMaker/data/test/smashedData/UWC_smashed_1m_patch*-*.tif",
    num_cols=25,
    num_rows=20
)



Skipping patch 0-0: Missing one or more files.
Skipping patch 0-1: Missing one or more files.
Skipping patch 0-2: Missing one or more files.
Skipping patch 0-3: Missing one or more files.
Skipping patch 0-4: Missing one or more files.
Skipping patch 0-5: Missing one or more files.
Skipping patch 0-6: Missing one or more files.
Skipping patch 0-7: Missing one or more files.
Skipping patch 0-8: Missing one or more files.
Skipping patch 0-9: Missing one or more files.
Skipping patch 0-10: Missing one or more files.
Skipping patch 0-11: Missing one or more files.
Skipping patch 0-12: Missing one or more files.
Skipping patch 0-13: Missing one or more files.
Skipping patch 0-14: Missing one or more files.
Skipping patch 0-15: Missing one or more files.
Skipping patch 0-16: Missing one or more files.
Skipping patch 0-17: Missing one or more files.
Skipping patch 0-18: Missing one or more files.
Skipping patch 0-19: Missing one or more files.
Skipping patch 1-0: Missing one or more files.
Ski